# Geometry-enriched BiLSTM: local training

Use the **Vector classification** kernel. Run cells in order.

**Shared input:** `(dx, dy, pen_up, centered_x, centered_y)`, constructed by exactly the same
`add_centered_coordinates` and `CachedSplit` code as the Transformer notebook.
Same cache, point normalization, 345 categories, exact sample-ID checks, seed 42,
5,000 / 500 / 500 sketches per class, and exclusion of sequences longer than 250 points.
No point-position or stroke-index embedding is added. The LSTM receives order through recurrence.

**Report configuration selected using validation, with added regularization and scheduling:** BiLSTM hidden size **512 per direction**,
one layer, final hidden state from each direction concatenated (1024-dimensional embedding),
linear 345-class head, **output dropout 0.1**, Adam lr 0.001, weight decay 0.0001, gradient clip 1.0,
batch size 128, 10 epochs. Float32; no warmup; validation-loss plateau scheduling by default.
The report's 76.26% is historical validation accuracy for stroke-3 inputs, not a predicted score
for this new five-feature model. These were the best validation settings among reported LSTMs;
they are not proven optimal for the enriched input.

The input width changes from 3 to 5; the historical model weights are not reused.
Both models train from scratch. Packed sequences exclude padding; last-state pooling is retained.
New results use `results/lstm_geometry_local_<timestamp>/`; test evaluation remains off by default.

The main comparison uses common data/features but model-specific training recipes:
LSTM uses the report recipe; Transformer uses AdamW, dropout, warmup/cosine and optional bfloat16.
Do not interpret runtime differences as architecture alone without matching precision/hardware.


The report recipe is now extended with output dropout, weight decay and a learning-rate scheduler. For the single LSTM layer, between-layer dropout stays disabled; dropout acts on the concatenated final hidden states before the classifier.

Connect to Weights & Biases in section 0 before training. The existing completed runs are preserved; each new run has a unique output folder and W&B run.

**New regularized trial:** output dropout 0.1, Adam weight decay **0.0001**, and
`ReduceLROnPlateau` on validation loss (factor 0.5, patience 1, minimum LR 0.00001).
This preserves Adam: its `weight_decay` is L2-style regularization, not AdamW's decoupled decay.
Set `CFG['scheduler']` to `'cosine'` for a fixed 10-epoch cosine schedule instead.
These are candidate settings, not guaranteed improvements. Existing run directories stay intact.


In [1]:
from pathlib import Path
from datetime import datetime
from contextlib import nullcontext
import copy, gzip, hashlib, json, math, random, time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data/experiment_345_seed42').is_dir()
             and (p / 'src').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the original vector-sketch-classification folder.')
CACHE = ROOT / 'data/experiment_345_seed42'
DATA = ROOT / 'data/processed_full'
REFERENCE_IDS = ROOT / 'results/lstm_baseline_reference_seed42'
LOCAL_LSTM = ROOT / 'results/bilstm_345_matched_seed42'

CFG = dict(seed=42, epochs=10, batch_size=128, lr=0.001, weight_decay=0.0001,
           warmup_fraction=0.0, grad_clip=1.0,
           hidden_dim=512, num_layers=1, bidirectional=True, dropout=0.1, max_length=250,
           representation='stroke3_centered_xy_v1', input_dim=5, optimizer='adam',
           scheduler='plateau', scheduler_factor=0.5, scheduler_patience=1,
           scheduler_min_lr=0.00001, scheduler_threshold=0.0001)
COORDINATE_SCALE = float(json.loads((DATA/'manifest.json').read_text())['normalization']['scale'])
if not math.isfinite(COORDINATE_SCALE) or COORDINATE_SCALE <= 0:
    raise ValueError('Invalid training-fitted coordinate scale.')
CFG['coordinate_scale'] = COORDINATE_SCALE
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Float32 on CPU and GPU; same precision as the report LSTM.
USE_BF16 = False  # Match the report LSTM's float32 training recipe.
torch.set_num_threads(4)
def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_all(CFG['seed'])
def autocast_context():
    return torch.autocast('cuda', dtype=torch.bfloat16) if USE_BF16 else nullcontext()
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))
def save_json(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, indent=2) + '\n', encoding='utf-8')
    temp.replace(path)
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
print('Project:', ROOT)
print('Device:', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU (full training will be slow)')
print('PyTorch:', torch.__version__, '| bfloat16:', USE_BF16)

Project: c:\Users\shrua\OneDrive\Desktop\11785 IDL\vector-sketch-classification
Device: NVIDIA GeForce RTX 5060 Laptop GPU
PyTorch: 2.11.0+cu128 | bfloat16: False


c:\Users\shrua\anaconda3\envs\vector-classification\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. Connect to Weights & Biases

First time only: the next cell installs `wandb` into this notebook's selected Python kernel
if it is missing. Set your project and optional username/team below, then run the connection cell.
Paste your API key into the **hidden prompt**, not into notebook source. It will not appear in
notebook outputs, saved configs, or notebook snapshots. Use the same project for both models.

`WANDB_MODE='online'` streams metrics to your account. Set `'offline'` only if you intentionally
want local W&B logs to sync later. No training starts in the connection cell.


In [2]:
import importlib.util
import sys
# Reuse the project-local SDK if available; otherwise install into the selected kernel.
wandb_sdk_dir = ROOT / '.local_packages/wandb_sdk'
if wandb_sdk_dir.is_dir() and str(wandb_sdk_dir) not in sys.path:
    sys.path.insert(0,str(wandb_sdk_dir))
if importlib.util.find_spec('wandb') is None:
    get_ipython().run_line_magic('pip', 'install wandb')


In [3]:
import wandb
import getpass
WANDB_PROJECT = 'sketch-classification'
WANDB_ENTITY = None     # Optional: your W&B username or team name.
WANDB_RUN_NAME = None   # Optional: a descriptive name; default is the unique output folder.
WANDB_MODE = 'online'
WANDB_READY = False
if WANDB_MODE == 'online':
    if not wandb.login(key=getpass.getpass('Paste your W&B API key (hidden): ')):
        raise RuntimeError('W&B login failed.')
elif WANDB_MODE != 'offline':
    raise ValueError("Choose 'online' or 'offline'.")
WANDB_READY = True
print('W&B configured:',WANDB_PROJECT,'| mode:',WANDB_MODE)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: C:\Users\shrua\_netrc
wandb: Currently logged in as: sihuilin (sihuilin-carnegie-mellon-university) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


W&B configured: sketch-classification | mode: online


## 1. Audit the existing splits

This reads IDs/metadata and hashes cache files, including the point arrays, to make the
training inputs traceable. It does not load the full point arrays into RAM. Split counts
must be 5,000 / 500 / 500 per category across all 345 categories. Missing data or mismatches
stop execution instead of silently drawing a new subset. Test inputs are audited here;
test predictions and metrics are computed only in the final-test cell.

In [4]:
def audit_data():
    manifest = read_json(DATA / 'manifest.json')
    if manifest['representation'] != 'stroke-3' or manifest['normalization']['fit_split'] != 'train':
        raise ValueError('Expected training-normalized stroke-3 data.')
    if manifest['archive_version'] != 'full':
        raise ValueError('Expected full archives.')
    categories = sorted(manifest['class_to_index'], key=manifest['class_to_index'].get)
    if len(categories) != 345:
        raise ValueError('Expected 345 categories.')
    source_hash = sha256(DATA / 'manifest.json')
    for split_name in ('train','valid','test'):
        cached_scale = read_json(CACHE/split_name/'metadata.json')['coordinate_scale']
        if float(cached_scale) != COORDINATE_SCALE:
            raise ValueError('Cache coordinate scale differs from manifest.')
    reference_cfg = read_json(REFERENCE_IDS / 'config.json')
    if reference_cfg['data']['categories'] != categories:
        raise ValueError('Category order differs from the earlier last-state run.')
    local_provenance = read_json(LOCAL_LSTM / 'provenance.json')
    audit, split_ids = {}, {}
    for split, per_class in [('train',5000), ('valid',500), ('test',500)]:
        folder = CACHE / split
        meta = read_json(folder / 'metadata.json')
        spec = meta['specification']
        expected = dict(source_manifest_sha256=source_hash, split=split,
                        per_class=per_class, max_length=250, seed=42, categories=categories)
        if any(spec.get(k) != v for k,v in expected.items()) or meta['scale_factor'] != 1.0:
            raise ValueError(f'{split}: cache preprocessing/sampling specification mismatch.')
        with gzip.open(folder / 'sample_ids.txt.gz', 'rt', encoding='utf-8') as f:
            ids = f.read().splitlines()
        if ids != read_json(REFERENCE_IDS / f'{split}_sample_ids.json'):
            raise ValueError(f'{split}: ordered sample IDs differ from the earlier last-state run.')
        ids_hash = hashlib.sha256(''.join(x+'\n' for x in ids).encode()).hexdigest()
        if len(ids) != len(set(ids)) or ids_hash != meta['sample_ids_sha256']:
            raise ValueError(f'{split}: invalid IDs/hash.')
        if split in ('train', 'valid'):
            earlier = local_provenance[split]
            for key in ('specification','samples','points','sample_ids_sha256','scale_factor','coordinate_scale','class_counts'):
                if meta[key] != earlier[key]:
                    raise ValueError(f'{split}: metadata differs from local LSTM: {key}')
        labels = np.load(folder / 'labels.npy', mmap_mode='r')
        offsets = np.load(folder / 'offsets.npy', mmap_mode='r')
        points = np.load(folder / 'points.npy', mmap_mode='r')
        lengths = np.diff(offsets)
        if len(labels) != len(ids) or len(ids) != 345*per_class or len(offsets) != len(ids)+1:
            raise ValueError(f'{split}: sample count mismatch.')
        if not np.issubdtype(labels.dtype, np.integer) or np.any(labels < 0) or np.any(labels >= 345):
            raise ValueError(f'{split}: invalid labels.')
        if not np.all(np.bincount(labels, minlength=345) == per_class):
            raise ValueError(f'{split}: class counts differ.')
        if offsets[0] != 0 or offsets[-1] != len(points) or points.shape != (meta['points'],3):
            raise ValueError(f'{split}: invalid point/offset arrays.')
        if points.dtype != np.float32 or np.any(lengths < 1) or np.any(lengths > 250):
            raise ValueError(f'{split}: invalid dtype/lengths.')
        # Scan in chunks: validate all features without loading a full split into RAM.
        for start in range(0, len(points), 1_000_000):
            chunk = points[start:start+1_000_000]
            if not np.isfinite(chunk).all() or not np.isin(chunk[:,2], [0,1]).all():
                raise ValueError(f'{split}: invalid stroke-3 features.')
        if any(not identifier.startswith(categories[int(label)]+'/full/'+split+'/')
               for identifier,label in zip(ids,labels)):
            raise ValueError(f'{split}: IDs and labels disagree.')
        hashes = {name:sha256(folder/name) for name in
                  ['points.npy','offsets.npy','labels.npy','sample_ids.txt.gz','metadata.json']}
        audit[split] = dict(samples=len(ids), sample_ids_sha256=ids_hash,
                            cache_files_sha256=hashes, metadata=meta)
        split_ids[split] = set(ids)
        print(f'{split}: {len(ids):,} sketches; exact earlier IDs verified')
    if any(split_ids[a] & split_ids[b] for a,b in [('train','valid'),('train','test'),('valid','test')]):
        raise ValueError('Splits overlap.')
    remote_hash = reference_cfg['data']['preprocessing']['manifest_sha256']
    audit['comparison'] = dict(local_lstm_metadata_matches=True, all_last_state_ids_match=True,
        current_manifest_sha256=source_hash, remote_manifest_sha256=remote_hash,
        remote_preprocessing_equivalence_certified=(source_hash == remote_hash))
    print('Local LSTM preprocessing metadata matches. Remote manifest matches:', source_hash == remote_hash)
    return categories, audit

CATEGORIES, DATA_AUDIT = audit_data()

train: 1,725,000 sketches; exact earlier IDs verified
valid: 172,500 sketches; exact earlier IDs verified
test: 172,500 sketches; exact earlier IDs verified
Local LSTM preprocessing metadata matches. Remote manifest matches: False


## 2. Identical five-feature batches for both encoders

The cache stays memory-mapped. A single function constructs both models' input tensors.
Centered coordinates use the training-fitted scale in the existing manifest, matching the
copy model's coordinate convention. No semantic labels or external observations are added.
Training shuffles identically for a given epoch/seed in both notebooks.


In [5]:
def add_centered_coordinates(strokes, lengths, coordinate_scale):
    """[B,T,3] -> [B,T,5]; identical implementation in both model notebooks."""
    if strokes.ndim != 3 or strokes.shape[-1] != 3:
        raise ValueError('Expected stroke-3 inputs.')
    b,t,_ = strokes.shape
    lengths = lengths.to(strokes.device)
    if lengths.shape != (b,) or torch.any(lengths < 1) or torch.any(lengths > t):
        raise ValueError('Invalid sketch lengths.')
    valid = torch.arange(t,device=strokes.device)[None,:] < lengths[:,None]
    clean = strokes.masked_fill(~valid[:,:,None],0)
    xy = clean[:,:,:2].cumsum(1) * coordinate_scale
    center = (xy * valid[:,:,None]).sum(1,keepdim=True) / lengths[:,None,None]
    centered = (xy-center).masked_fill(~valid[:,:,None],0)
    return torch.cat([clean,centered],dim=-1)

class CachedSplit:
    def __init__(self, split):
        folder = CACHE / split
        self.points = np.load(folder/'points.npy', mmap_mode='r')
        self.offsets = np.load(folder/'offsets.npy', mmap_mode='r')
        self.labels = np.load(folder/'labels.npy', mmap_mode='r')
        self.lengths = np.diff(self.offsets)
    def __len__(self):
        return len(self.labels)
    def batch(self, indices):
        indices = np.asarray(indices, dtype=np.int64)
        lengths = self.lengths[indices]
        steps = np.arange(int(lengths.max()))[None,:]
        valid = steps < lengths[:,None]
        gather = self.offsets[indices,None] + np.minimum(steps, lengths[:,None]-1)
        strokes = np.asarray(self.points[gather]).copy()
        strokes[~valid] = 0
        features = add_centered_coordinates(torch.from_numpy(strokes).to(DEVICE),
                                            torch.from_numpy(lengths.copy()).long().to(DEVICE), COORDINATE_SCALE)
        return (features,
                torch.from_numpy(lengths.copy()).long().to(DEVICE),
                torch.from_numpy(np.asarray(self.labels[indices]).copy()).long().to(DEVICE))
    def batches(self, batch_size, shuffle=False, seed=42):
        order = np.random.default_rng(seed).permutation(len(self)) if shuffle else np.arange(len(self))
        for start in range(0,len(self),batch_size):
            yield self.batch(order[start:start+batch_size])

train_data, valid_data = CachedSplit('train'), CachedSplit('valid')

## 3. Report-selected BiLSTM on the shared representation

Each point has the same five features supplied to the Transformer. A bidirectional LSTM
reads the full sketch in both directions. Packed sequences prevent padding from advancing
its hidden states. Concatenate the final forward/backward states of the top layer and apply
a linear classifier. No additional feature projection or positional embedding is needed.


In [6]:
from torch.nn.utils.rnn import pack_padded_sequence

class LSTMEncoder(nn.Module):
    def __init__(self, hidden_dim=512, num_layers=1, bidirectional=True, dropout=0.0, max_length=250):
        super().__init__()
        self.max_length = max_length
        self.num_directions = 2 if bidirectional else 1
        self.embedding_dim = hidden_dim*self.num_directions
        self.lstm = nn.LSTM(5,hidden_dim,num_layers=num_layers,bidirectional=bidirectional,
                            batch_first=True,dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
    def forward(self, strokes, lengths):
        if strokes.ndim != 3 or strokes.shape[-1] != 5:
            raise ValueError('Expected shared features [B,T,5].')
        b,t,_ = strokes.shape
        if not 1 <= t <= self.max_length or lengths.shape != (b,):
            raise ValueError('Invalid sequence width or lengths shape.')
        if lengths.dtype not in (torch.int32,torch.int64) or torch.any(lengths < 1) or torch.any(lengths > t):
            raise ValueError('Lengths must be integer values in [1,T].')
        packed = pack_padded_sequence(strokes,lengths.cpu(),batch_first=True,enforce_sorted=False)
        _,(h_n,_) = self.lstm(packed)
        pooled = h_n[-self.num_directions:].transpose(0,1).reshape(b,-1)
        return self.dropout(pooled)

class SketchClassifier(nn.Module):
    def __init__(self,num_classes=345,**encoder_kwargs):
        super().__init__()
        self.encoder = LSTMEncoder(**encoder_kwargs)
        self.head = nn.Linear(self.encoder.embedding_dim,num_classes)
    def embed(self,strokes,lengths):
        return self.encoder(strokes,lengths)
    def forward(self,strokes,lengths):
        return self.head(self.embed(strokes,lengths))

MODEL_CFG = {k:CFG[k] for k in ['hidden_dim','num_layers','bidirectional','dropout','max_length']}
def make_model(num_classes=345):
    return SketchClassifier(num_classes,**MODEL_CFG).to(DEVICE)
model = make_model()
print(model)
print('Trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))


SketchClassifier(
  (encoder): LSTMEncoder(
    (lstm): LSTM(5, 512, batch_first=True, bidirectional=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (head): Linear(in_features=1024, out_features=345, bias=True)
)
Trainable parameters: 2479449


## 4. Quick synthetic checks

Checks shapes, padding invariance, mixed-length batches, boundary lengths, finite gradients,
checkpoint reload and learning on a tiny synthetic batch. This is not a QuickDraw score.
The sanity model is discarded before training.

In [7]:
def model_checks():
    # Shared representation checks: hand-computed geometry and padding invariance.
    raw = torch.tensor([[[1.,0.,0.],[1.,0.,1.],[99.,99.,0.]]],device=DEVICE)
    lens = torch.tensor([2],device=DEVICE)
    features = add_centered_coordinates(raw,lens,1.0)
    expected = torch.tensor([[[1.,0.,0.,-0.5,0.],[1.,0.,1.,0.5,0.],[0.,0.,0.,0.,0.]]],device=DEVICE)
    torch.testing.assert_close(features,expected)
    extended_features = add_centered_coordinates(F.pad(raw,(0,0,0,4),value=999),lens,1.0)
    torch.testing.assert_close(features,extended_features[:,:3])
    torch.testing.assert_close(extended_features[:,3:],torch.zeros_like(extended_features[:,3:]))
    seed_all(7)
    probe = make_model(3).eval()
    x = torch.randn(3,9,5,device=DEVICE)
    lengths = torch.tensor([9,5,1],device=DEVICE)
    with torch.no_grad():
        original = probe(x,lengths)
        assert original.shape == (3,3)
        extended = F.pad(x,(0,0,0,6),value=123.0)
        torch.testing.assert_close(original,probe(extended,lengths),atol=2e-5,rtol=2e-5)
        changed = x.clone()
        changed[1,5:] = -999; changed[2,1:] = 999
        torch.testing.assert_close(original,probe(changed,lengths),atol=2e-5,rtol=2e-5)
        torch.testing.assert_close(original[1:2],probe(x[1:2,:5],lengths[1:2]),atol=2e-5,rtol=2e-5)
        assert probe(torch.zeros(1,250,5,device=DEVICE),torch.tensor([250],device=DEVICE)).shape == (1,3)
        for bad in [torch.tensor([0,5,1],device=DEVICE),torch.tensor([10,5,1],device=DEVICE)]:
            try: probe(x,bad)
            except ValueError: pass
            else: raise AssertionError('Invalid lengths accepted')
    # In-memory torch serialization verifies state-dict save/load behavior.
    import io
    buffer = io.BytesIO()
    torch.save(probe.state_dict(),buffer); buffer.seek(0)
    restored = make_model(3).eval()
    restored.load_state_dict(torch.load(buffer,map_location=DEVICE,weights_only=True))
    with torch.no_grad():
        torch.testing.assert_close(original,restored(x,lengths))
    probe.train()  # cuDNN LSTM backward requires a training-mode forward.
    labels = torch.arange(3,device=DEVICE)
    optimizer = torch.optim.AdamW(probe.parameters(),lr=0.002)
    # Disable output dropout only in this synthetic learning check; the real model retains 0.1.
    probe.encoder.dropout.p = 0.0
    before = float(F.cross_entropy(probe(x,lengths),labels).detach())
    for _ in range(60):
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(probe(x,lengths),labels)
        loss.backward()
        assert all(torch.isfinite(p.grad).all() for p in probe.parameters() if p.grad is not None)
        torch.nn.utils.clip_grad_norm_(probe.parameters(),1.0)
        optimizer.step()
    with torch.no_grad():
        after = float(F.cross_entropy(probe(x,lengths),labels))
        assert after < before*0.3 and torch.equal(probe(x,lengths).argmax(1),labels)
    print(f'PASS: model checks; synthetic loss {before:.4f} -> {after:.4f}')
model_checks()
seed_all(CFG['seed'])
model = make_model()

PASS: model checks; synthetic loss 1.1007 -> 0.0002


## 5. Training and checkpoints

Adam starts at lr 0.001 with weight decay 0.0001 (L2-style), output dropout 0.1,
batch 128, gradient clipping 1.0, 10 epochs, cross-entropy and float32.

**Default: `scheduler='plateau'`.** After validation, call `ReduceLROnPlateau` on validation
loss. Factor 0.5 and patience 1 mean the LR halves after two consecutive epochs without a
significant loss improvement. Minimum LR is 0.00001.

**Alternative: `scheduler='cosine'`.** CosineAnnealingLR decays the LR toward 0.00001 over
the configured epoch budget, regardless of validation performance. There is no warmup.
Choose in the configuration cell, then rerun setup/model/helper cells before starting a new run.

The history and W&B log both the learning rate **used for the completed epoch** and the rate
scheduled for the **next epoch**. Checkpoints include scheduler state. `best.pt` is selected
by validation accuracy; the scheduler monitors validation loss. Test metrics are not sent to W&B.


In [8]:
def synchronize():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()

def run_epoch(model, data, optimizer=None, epoch=0, step=0, total_steps=1):
    training = optimizer is not None
    model.train(training)
    warmup = max(1,round(total_steps*CFG['warmup_fraction']))
    sums = torch.zeros(4,device=DEVICE,dtype=torch.float64)
    synchronize(); start = time.perf_counter()
    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        batches = data.batches(CFG['batch_size'],shuffle=training,seed=CFG['seed']+epoch)
        for x,lengths,labels in tqdm(batches,total=math.ceil(len(data)/CFG['batch_size']),
                                     desc=f'{"train" if training else "valid/test"} {epoch}',leave=False):
            if training:
                optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                logits = model(x,lengths)
                loss = F.cross_entropy(logits,labels)
            if not torch.isfinite(loss): raise FloatingPointError('Nonfinite loss.')
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),CFG['grad_clip'],error_if_nonfinite=True)
                optimizer.step(); step += 1
            top = logits.topk(min(5,logits.shape[1]),dim=1).indices
            sums += torch.stack([loss.detach().double()*len(labels),
                                 (top[:,0]==labels).sum().double(),
                                 (top==labels[:,None]).any(1).sum().double(),
                                 torch.tensor(len(labels),device=DEVICE,dtype=torch.float64)])
    synchronize(); elapsed = time.perf_counter()-start
    loss_sum,correct,top5,count = sums.tolist()
    return dict(loss=loss_sum/count,acc=correct/count,top5_acc=top5/count,
                samples=int(count),seconds=elapsed),step

def save_checkpoint(path, payload):
    temp = Path(path).with_suffix('.tmp')
    torch.save(payload,temp); temp.replace(path)

def build_scheduler(optimizer, cfg):
    if not 0 < cfg['scheduler_min_lr'] <= cfg['lr']:
        raise ValueError('scheduler_min_lr must be positive and no larger than the initial LR.')
    if cfg['scheduler'] == 'plateau':
        if not 0 < cfg['scheduler_factor'] < 1 or cfg['scheduler_patience'] < 0:
            raise ValueError('Invalid plateau scheduler settings.')
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,mode='min',factor=cfg['scheduler_factor'],
            patience=cfg['scheduler_patience'],threshold=cfg['scheduler_threshold'],
            min_lr=cfg['scheduler_min_lr'])
    if cfg['scheduler'] == 'cosine':
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,T_max=cfg['epochs'],eta_min=cfg['scheduler_min_lr'])
    raise ValueError("scheduler must be 'plateau' or 'cosine'.")

def train_local():
    """Start a new model/optimizer at epoch 1 and log one W&B record per epoch."""
    if 'wandb' not in globals() or not globals().get('WANDB_READY',False):
        raise RuntimeError('Run section 0: Weights & Biases connection before training.')
    seed_all(CFG['seed'])
    model = make_model()
    optimizer = torch.optim.Adam(model.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'])
    scheduler = build_scheduler(optimizer,CFG)
    out = ROOT/'results'/('lstm_geometry_wandb_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    out.mkdir(parents=True,exist_ok=False)
    save_json(out/'config.json',CFG); save_json(out/'data_audit.json',DATA_AUDIT)
    precision='bfloat16 autocast' if USE_BF16 else 'float32'
    provenance=dict(torch=torch.__version__,numpy=np.__version__,device=str(DEVICE),
        gpu=torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,
        precision=precision,parameters=sum(p.numel() for p in model.parameters()),
        categories=CATEGORIES,architecture='stroke3-centered-xy-bilstm512-last-outputdrop01',
        representation=CFG['representation'],coordinate_scale=COORDINATE_SCALE,
        test_used_for_selection=False,initialization='fresh random weights, no checkpoint loaded')
    save_json(out/'provenance.json',provenance)
    notebook_path=ROOT/'notebooks/lstm_geometry_local.ipynb'
    if notebook_path.exists(): (out/'notebook_snapshot.ipynb').write_bytes(notebook_path.read_bytes())
    print('New run, starting at epoch 1. Outputs:',out,flush=True)
    history,step,best=[],0,-1.0
    total_steps=CFG['epochs']*math.ceil(len(train_data)/CFG['batch_size'])
    with wandb.init(project=WANDB_PROJECT,entity=WANDB_ENTITY,mode=WANDB_MODE,
            name=WANDB_RUN_NAME or out.name,group='geometry-model-comparison',
            tags=['lstm','geometry-5features','from-scratch'],dir=str(out),
            config=dict(CFG,model='lstm',precision=precision,
                        parameters=provenance['parameters']),
            settings=wandb.Settings(save_code=False,disable_git=True)) as wb_run:
        wb_run.define_metric('epoch')
        for namespace in ['train/*','valid/*','optimizer/*','runtime/*']:
            wb_run.define_metric(namespace,step_metric='epoch')
        wb_run.summary['local_output_dir']=str(out)
        save_json(out/'wandb_run.json',dict(id=wb_run.id,url=wb_run.url,
            project=WANDB_PROJECT,entity=wb_run.entity,mode=WANDB_MODE))
        for epoch in range(1,CFG['epochs']+1):
            if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats()
            epoch_lr=optimizer.param_groups[0]['lr']
            train_metrics,step=run_epoch(model,train_data,optimizer,epoch,step,total_steps)
            valid_metrics,_=run_epoch(model,valid_data,epoch=epoch)
            if CFG['scheduler']=='plateau':
                scheduler.step(valid_metrics['loss'])
            else:
                scheduler.step()
            next_lr=optimizer.param_groups[0]['lr']
            peak_memory=torch.cuda.max_memory_allocated() if DEVICE.type=='cuda' else None
            history.append(dict(epoch=epoch,train=train_metrics,valid=valid_metrics,
                                peak_gpu_memory_bytes=peak_memory,learning_rate=epoch_lr,next_learning_rate=next_lr))
            improved=valid_metrics['acc']>best
            best=max(best,valid_metrics['acc'])
            payload=dict(epoch=epoch,step=step,best_acc=best,config=CFG,
                model_state=model.state_dict(),optimizer_state=optimizer.state_dict(),
                scheduler_state=scheduler.state_dict(),history=history,
                wandb_run_id=wb_run.id,
                python_rng=random.getstate(),numpy_rng=np.random.get_state(),torch_rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state_all() if DEVICE.type=='cuda' else [])
            if improved: save_checkpoint(out/'best.pt',payload)
            save_checkpoint(out/'last.pt',payload)
            save_json(out/'history.json',history)
            log=dict(epoch=epoch,**{'train/loss':train_metrics['loss'],
                'train/accuracy':train_metrics['acc'],'train/top5_accuracy':train_metrics['top5_acc'],
                'valid/loss':valid_metrics['loss'],'valid/accuracy':valid_metrics['acc'],
                'valid/top5_accuracy':valid_metrics['top5_acc'],
                'optimizer/learning_rate':epoch_lr,
                'optimizer/next_learning_rate':next_lr,
                'runtime/train_seconds':train_metrics['seconds'],
                'runtime/valid_seconds':valid_metrics['seconds']})
            if peak_memory is not None: log['runtime/peak_gpu_memory_mb']=peak_memory/(1024**2)
            wb_run.log(log)
            wb_run.summary['best_valid_accuracy']=best
            wb_run.summary['best_epoch']=max(history,key=lambda r:r['valid']['acc'])['epoch']
            wb_run.summary['completed_epochs']=epoch
            print(f"Epoch {epoch:02d}/{CFG['epochs']} | train {train_metrics['acc']:.2%} | "
                  f"valid {valid_metrics['acc']:.2%} | best {best:.2%} | LR {epoch_lr:.6g} -> {next_lr:.6g}",flush=True)
    return out


## 6. Train from scratch with W&B logging

Run the connection cell in section 0 first, then the setup/audit/model/helper cells above.
This training cell always initializes a new model and optimizer at epoch 1. It creates a new
local results directory and W&B run each time. It does not resume any earlier checkpoint.
Local `best.pt` and `last.pt` are saved every epoch as appropriate. W&B records configuration,
train/validation loss, top-1/top-5 accuracy, learning rate, timing, and best validation epoch.
Accuracy values in W&B are fractions (0.78 = 78%). Checkpoints stay local.


Weight decay and the scheduler apply only to the next fresh run; they do not change an already running kernel or experiment.

In [9]:
RUN_DIR = train_local()

New run, starting at epoch 1. Outputs: c:\Users\shrua\OneDrive\Desktop\11785 IDL\vector-sketch-classification\results\lstm_geometry_wandb_20261101_200003_200378


Epoch 01/10 | train 52.27% | valid 65.05% | best 65.05% | LR 0.001 -> 0.001


Epoch 02/10 | train 66.33% | valid 67.62% | best 67.62% | LR 0.001 -> 0.001


Epoch 03/10 | train 68.10% | valid 68.99% | best 68.99% | LR 0.001 -> 0.001


Epoch 04/10 | train 68.87% | valid 69.55% | best 69.55% | LR 0.001 -> 0.001


Epoch 05/10 | train 69.32% | valid 70.05% | best 70.05% | LR 0.001 -> 0.001


Epoch 06/10 | train 69.59% | valid 70.09% | best 70.09% | LR 0.001 -> 0.001


Epoch 07/10 | train 69.77% | valid 70.38% | best 70.38% | LR 0.001 -> 0.001


Traceback (most recent call last):                           
  File "C:\Users\shrua\AppData\Local\Temp\ipykernel_33968\2938107941.py", line 93, in train_local
    train_metrics,step=run_epoch(model,train_data,optimizer,epoch,step,total_steps)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\shrua\AppData\Local\Temp\ipykernel_33968\2938107941.py", line 22, in run_epoch
    loss.backward()
  File "c:\Users\shrua\anaconda3\envs\vector-classification\Lib\site-packages\torch\_tensor.py", line 631, in backward
    torch.autograd.backward(
  File "c:\Users\shrua\anaconda3\envs\vector-classification\Lib\site-packages\torch\autograd\__init__.py", line 381, in backward
    _engine_run_backward(
  File "c:\Users\shrua\anaconda3\envs\vector-classification\Lib\site-packages\torch\autograd\graph.py", line 869, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
         

epoch,▁▂▃▅▆▇█
optimizer/learning_rate,▁▁▁▁▁▁▁
optimizer/next_learning_rate,▁▁▁▁▁▁▁
runtime/peak_gpu_memory_mb,▁▁▁▁▁▁▁
runtime/train_seconds,▁▅██▆▆▇
runtime/valid_seconds,▅█▇▆▆▅▁
train/accuracy,▁▇▇████
train/loss,█▂▂▁▁▁▁
train/top5_accuracy,▁▇█████
valid/accuracy,▁▄▆▇███
+2,...


KeyboardInterrupt: 

## 7. Learning curves and best-checkpoint validation

In [ ]:
history = read_json(RUN_DIR/'history.json')
epochs = [r['epoch'] for r in history]
fig, axes = plt.subplots(1,2,figsize=(11,4))
for split in ['train','valid']:
    axes[0].plot(epochs,[r[split]['loss'] for r in history],label=split)
    axes[1].plot(epochs,[100*r[split]['acc'] for r in history],label=split)
axes[0].set_ylabel('Cross-entropy'); axes[1].set_ylabel('Top-1 accuracy (%)')
for ax in axes: ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.2)
fig.tight_layout(); fig.savefig(RUN_DIR/'curves.png',dpi=160); plt.show()
selected = torch.load(RUN_DIR/'best.pt',map_location='cpu',weights_only=False)
model = make_model(); model.load_state_dict(selected['model_state'])
validation,_ = run_epoch(model,valid_data)
if abs(validation['acc']-selected['best_acc']) > 1e-8:
    raise AssertionError('Reloaded validation accuracy differs from checkpoint selection.')
save_json(RUN_DIR/'metrics_valid.json',dict(**validation,epoch=selected['epoch'],split='valid'))
print('Selected epoch:',selected['epoch'],'| validation:',validation)

## 8. Final held-out test (off by default)

Set `RUN_FINAL_TEST=True` only after architecture/training settings are frozen.
This reloads the validation-selected checkpoint and saves overall top-1/top-5, loss,
per-class accuracy and predictions in source order. Do not tune settings using test results.
Existing metrics are displayed instead of repeating the evaluation.

In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    if (RUN_DIR/'metrics_test.json').exists():
        print('Already evaluated:',read_json(RUN_DIR/'metrics_test.json'))
    else:
        selected = torch.load(RUN_DIR/'best.pt',map_location='cpu',weights_only=False)
        model = make_model(); model.load_state_dict(selected['model_state']); model.eval()
        test_data = CachedSplit('test')
        loss_sum=correct=correct5=count=0
        per_count=np.zeros(345,dtype=np.int64); per_correct=np.zeros(345,dtype=np.int64)
        predictions=[]
        synchronize(); start=time.perf_counter()
        with torch.inference_mode():
            for x,lengths,labels in tqdm(test_data.batches(CFG['batch_size']),
                                         total=math.ceil(len(test_data)/CFG['batch_size'])):
                with autocast_context():
                    logits=model(x,lengths)
                top=logits.topk(5,dim=1).indices
                loss_sum += float(F.cross_entropy(logits.float(),labels,reduction='sum'))
                correct += int((top[:,0]==labels).sum())
                correct5 += int((top==labels[:,None]).any(1).sum()); count += len(labels)
                y=labels.cpu().numpy(); pred=top.cpu().numpy(); predictions.append(pred)
                per_count += np.bincount(y,minlength=345)
                per_correct += np.bincount(y[pred[:,0]==y],minlength=345)
        synchronize()
        metrics=dict(split='test',epoch=selected['epoch'],samples=count,loss=loss_sum/count,
                     acc=correct/count,top5_acc=correct5/count,seconds=time.perf_counter()-start,
                     checkpoint_sha256=sha256(RUN_DIR/'best.pt'),
                     sample_ids_sha256=DATA_AUDIT['test']['sample_ids_sha256'])
        np.savez_compressed(RUN_DIR/'test_predictions.npz',top5=np.concatenate(predictions),
                            labels=np.asarray(test_data.labels))
        save_json(RUN_DIR/'per_class_test.json',[
            dict(category=name,samples=int(per_count[i]),acc=float(per_correct[i]/per_count[i]))
            for i,name in enumerate(CATEGORIES)])
        save_json(RUN_DIR/'metrics_test.json',metrics)
        print(metrics)
else:
    print('Test evaluation is off. Freeze settings, then set RUN_FINAL_TEST=True.')

## 9. Compare the two enriched models

Both models receive identical five-feature tensors. The LSTM preserves sequence order through
recurrence; the Transformer uses fixed positional encoding and full self-attention. LSTM reads
5 features directly into its gates; Transformer learns a 5-to-128 projection. These are model
components, not differences in the observed sketch input. The comparison is not parameter-matched.

| Setting | BiLSTM | Transformer |
|---|---|---|
| Shared input | dx, dy, pen_up, centered_x, centered_y | Same |
| Layers | 1 bidirectional | 4 attention blocks |
| Width | 512 per direction | 128; 4 heads; FFN 384 |
| Summary | Final forward/backward hidden states | CLS token |
| Parameters | 2,479,449 | 707,161 |
| Optimizer / initial LR | Adam / 0.001 | AdamW / 0.0003 |
| Weight decay | 0.0001 (Adam L2-style) | 0.01 (AdamW) |
| Dropout | 0.1 on output embedding | 0.1 residual/FFN |
| Precision | Float32 | bfloat16 when supported |
| Budget | 10 epochs, batch 128 | 20 epochs, batch 128 |

The cell below reads the latest completed run of each model, verifies identical data audits,
and compares validation-selected checkpoints. It does not train or evaluate test data.
Test metrics appear only if you already ran the final-test cells. Use validation for decisions.


The comparison now uses the new W&B runs. Their epoch budgets differ (10 versus 20); report that difference rather than describing this as an equal-budget comparison.

LSTM now uses validation-loss plateau scheduling by default, or selectable cosine annealing; Transformer retains warmup/cosine.

In [ ]:
def latest_completed(pattern):
    for folder in sorted((ROOT/'results').glob(pattern),reverse=True):
        if not (folder/'history.json').exists(): continue
        config = read_json(folder/'config.json')
        history = read_json(folder/'history.json')
        if len(history) == config['epochs'] and (folder/'best.pt').exists():
            return folder
    return None

comparison_runs = {'BiLSTM':latest_completed('lstm_geometry_wandb_*'),
                   'Transformer':latest_completed('transformer_geometry_wandb_*')}
if all(comparison_runs.values()):
    left,right = list(comparison_runs.values())
    if read_json(left/'data_audit.json') != read_json(right/'data_audit.json'):
        raise ValueError('The runs have different data audits; do not treat them as a matched comparison.')
    configs = [read_json(p/'config.json') for p in comparison_runs.values()]
    for key in ['representation','input_dim','coordinate_scale','seed','batch_size']:
        if configs[0][key] != configs[1][key]:
            raise ValueError(f'Comparison differs on {key}.')
    if configs[0]['epochs'] != configs[1]['epochs']:
        print('Different epoch budgets: LSTM 10, Transformer 20; report this explicitly.')
    print(f"{'Model':<14} {'Best epoch':>10} {'Val top-1':>12} {'Test top-1':>12} {'Train min':>12}")
    for name,folder in comparison_runs.items():
        history = read_json(folder/'history.json')
        best = max(history,key=lambda row:row['valid']['acc'])
        test = read_json(folder/'metrics_test.json') if (folder/'metrics_test.json').exists() else None
        test_text = f"{100*test['acc']:.2f}%" if test else 'not run'
        minutes = sum(row['train']['seconds'] for row in history)/60
        print(f"{name:<14} {best['epoch']:>10} {100*best['valid']['acc']:>11.2f}% {test_text:>12} {minutes:>12.1f}")
        print('  ',folder)
else:
    print('Finish a full run of both notebooks first. Available runs:',comparison_runs)
